# Fast Charge Profile Generation with Anode Potential Riding

This notebook demonstrates fast charging with anode potential protection:
1. Start at 10% SOC
2. Charge at high C-rate until anode potential threshold is reached
3. Ride the anode potential plateau until one of:
   - Maximum charge time
   - Upper voltage threshold
   - Maximum temperature threshold
4. CV hold until termination current

## Overview
- **Cell**: Tesla Model 3 Prismatic 160Ah
- **Initial SOC**: 10%
- **Charge Rate**: 10C (constant current phase)
- **Anode Potential Threshold**: 0.02V (lithium plating protection)

In [1]:
# Import required libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from model_library import run_spmet

## 1. Load Cell Parameters

In [2]:
# manifest_path = Path("../cells/Tesla_Model3_Prismatic_160Ah_manifest.json")
manifest_path = Path("../cells/VW_ID3_Pouch_80Ah_manifest.json")

material_path = Path("../materials")

with open(manifest_path, "r") as f:
    cell_design_manifest = json.load(f)

    # Load material properties
    for component in ["separator", "electrolyte"]:
        material_name = cell_design_manifest["cell_design"][component]["material"][
            "type"
        ]
        with open(material_path / f"{material_name}.json", "r") as mf:
            cell_design_manifest["cell_design"][component]["material"] = json.load(mf)

    for component in ["negative_electrode", "positive_electrode"]:
        material_name = cell_design_manifest["cell_design"][component]["coating"][
            "formulation"
        ]["primary_active_material"]["name"]
        with open(material_path / f"{material_name}.json", "r") as mf:
            cell_design_manifest["cell_design"][component]["material"] = json.load(mf)

cell_design = cell_design_manifest["cell_design"]
cell_design["nominal_capacity"] = {
    "value": cell_design_manifest["kpis"]["nominal_capacity"]["value"],
    "unit": "Ah",
}
cell_design["nominal_energy"] = {
    "value": cell_design_manifest["kpis"]["nominal_energy"]["value"],
    "unit": "Wh",
}
cell_design["cell_volume"] = {
    "value": cell_design_manifest["kpis"]["cell_volume"]["value"],
    "unit": "L",
}
cell_design["nominal_voltage"] = {
    "value": cell_design_manifest["kpis"]["nominal_voltage"]["value"],
    "unit": "V",
}
upper_voltage_cutoff = cell_design["upper_voltage_cutoff"][
    "value"
]
lower_voltage_cutoff = cell_design["lower_voltage_cutoff"][
    "value"
]
# Load duty cycle
print(f"Cell: {cell_design_manifest['metadata']['name']}")
print(f"Form factor: {cell_design['form_factor']}")
print(f"Capacity: {cell_design['nominal_capacity']['value']:.1f} Ah")
print(f"Energy: {cell_design['nominal_energy']['value']:.1f} Wh")
print(
    f"Nominal voltage: {cell_design_manifest['kpis']['nominal_voltage']['value']:.2f} V"
)
print(
    f"Voltage range: {cell_design['lower_voltage_cutoff']['value']:.2f} - {cell_design['upper_voltage_cutoff']['value']:.2f} V"
)

Cell: VW ID3 Pouch 80Ah
Form factor: Pouch
Capacity: 80.5 Ah
Energy: 295.4 Wh
Nominal voltage: 3.67 V
Voltage range: 2.80 - 4.20 V


## 2. Fast Charge Configuration

In [ ]:
# Fast charge parameters
INITIAL_SOC = 0.10  # 10% SOC
CHARGE_C_RATE = 10.0  # 10C charging
ANODE_POTENTIAL_THRESHOLD_V = 0.02  # Lithium plating protection threshold
MAX_CHARGE_TIME_S = 60 * 60  # 60 minutes maximum
MAX_TEMPERATURE_K = 273.15 + 45  # 45C maximum temperature

print(f"Fast Charge Configuration:")
print(f"  Initial SOC: {INITIAL_SOC*100:.0f}%")
print(
    f"  Charge C-rate: {CHARGE_C_RATE}C ({CHARGE_C_RATE * cell_design['nominal_capacity']['value']:.1f} A)"
)
print(f"  Anode potential threshold: {ANODE_POTENTIAL_THRESHOLD_V} V")
print(f"  Max charge time: {MAX_CHARGE_TIME_S/60:.0f} minutes")
print(f"  Max temperature: {MAX_TEMPERATURE_K - 273.15:.0f} C")
print(f"  Upper voltage cutoff: {upper_voltage_cutoff} V")

Fast Charge Configuration:
  Initial SOC: 10%


NameError: name 'nominal_capacity' is not defined

## 3. Run Fast Charge Simulation

Use `run_spmet_manifest` with `ride_anode_potential=True` to enable the 3-phase fast charge protocol:
1. CC phase at specified C-rate until anode potential threshold
2. Anode potential riding phase (maintains constant anode potential)
3. CV phase until termination current

In [ ]:
# Fast charge simulation config
fast_charge_config = {
    # Thermal conditions
    "ambient_temperature": 298.15,  # 25C
    "initial_temperature": 298.15,  # 25C
    "total_heat_transfer_coefficient": 1,
    "cooling_surface_area": 0.001,
    
    # Voltage limits
    "upper_voltage_cutoff": upper_voltage_cutoff,
    "lower_voltage_cutoff": lower_voltage_cutoff,
    "contact_resistance": 0.0001,
    
    # Initial state
    "initial_soc": INITIAL_SOC,
    
    # Fast charge specific settings
    "ride_anode_potential": True,  # Enable anode potential riding
    "anode_potential_threshold_V": ANODE_POTENTIAL_THRESHOLD_V,
    "jelly_roll_temperature_threshold_K": MAX_TEMPERATURE_K,
    "max_charge_time_s": MAX_CHARGE_TIME_S,
    "cv_termination_c_rate": 0.05,  # C/20 termination
    
    # Experiment definition
    "experiments": [f"Charge at {CHARGE_C_RATE}C for {MAX_CHARGE_TIME_S} seconds"],
    "experiment_labels": [f"{CHARGE_C_RATE}C_fast_charge"],
    "period": "1 second",
    "Current function [A]": CHARGE_C_RATE * cell_design["nominal_capacity"]["value"],
}

# Run simulation
results = run_spmet(cell_design, fast_charge_config)

## 4. Analyze Results

In [ ]:
# Extract results
r = results[0]

if r['success']:
    time_s = r['time_s']
    voltage_V = r['voltage_V']
    current_A = r['current_A']
    anode_potential_V = r['anode_potential_V']
    temperature_K = r['temperature_K']
    
    # Calculate metrics
    total_time_min = (time_s[-1] - time_s[0]) / 60
    max_temp_C = np.max(temperature_K) - 273.15
    min_anode_potential_V = np.min(anode_potential_V)
    charge_capacity_Ah = np.trapezoid(-current_A, time_s) / 3600
    
    print(f"\nFast Charge Results:")
    print(f"  Total charge time: {total_time_min:.1f} minutes")
    print(f"  Charge capacity: {charge_capacity_Ah:.1f} Ah")
    print(f"  Final voltage: {voltage_V[-1]:.3f} V")
    print(f"  Max temperature: {max_temp_C:.1f} C")
    print(f"  Min anode potential: {min_anode_potential_V:.4f} V")
else:
    print(f"Simulation failed: {r.get('error', 'Unknown error')}")

In [ ]:
if r['success']:
    # Create comprehensive plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    time_min = (time_s - time_s[0]) / 60

    # Plot 1: Voltage vs Time
    ax1 = axes[0, 0]
    ax1.plot(time_min, voltage_V, 'b-', linewidth=2)
    ax1.axhline(y=upper_voltage_cutoff, color='r', linestyle='--', label=f'Upper limit ({upper_voltage_cutoff}V)')
    ax1.set_xlabel('Time [min]')
    ax1.set_ylabel('Voltage [V]')
    ax1.set_title('Voltage vs Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Plot 2: Current vs Time
    ax2 = axes[0, 1]
    ax2.plot(time_min, -current_A, 'g-', linewidth=2)  # Negative for charge current display
    ax2.set_xlabel('Time [min]')
    ax2.set_ylabel('Charge Current [A]')
    ax2.set_title('Charge Current vs Time')
    ax2.grid(True, alpha=0.3)

    # Plot 3: Anode Potential vs Time
    ax3 = axes[1, 0]
    ax3.plot(time_min, anode_potential_V, 'm-', linewidth=2)
    ax3.axhline(y=ANODE_POTENTIAL_THRESHOLD_V, color='r', linestyle='--', 
                label=f'Threshold ({ANODE_POTENTIAL_THRESHOLD_V}V)')
    ax3.axhline(y=0, color='k', linestyle=':', alpha=0.5, label='Li plating (0V)')
    ax3.set_xlabel('Time [min]')
    ax3.set_ylabel('Anode Potential [V]')
    ax3.set_title('Anode Potential vs Time')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # Plot 4: Temperature vs Time
    ax4 = axes[1, 1]
    temp_C = temperature_K - 273.15
    ax4.plot(time_min, temp_C, 'r-', linewidth=2)
    ax4.axhline(y=MAX_TEMPERATURE_K - 273.15, color='r', linestyle='--', 
                label=f'Max limit ({MAX_TEMPERATURE_K - 273.15:.0f}C)')
    ax4.set_xlabel('Time [min]')
    ax4.set_ylabel('Temperature [C]')
    ax4.set_title('Temperature vs Time')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.suptitle(f'Fast Charge Profile: {CHARGE_C_RATE}C from {INITIAL_SOC*100:.0f}% SOC', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Compare Multiple C-rates

In [ ]:
# Run simulations at different C-rates
c_rates = [1, 2, 4, 6, 8, 10]
all_results = {}

for c_rate in c_rates:
    config = {
        **fast_charge_config,
        "experiments": [f"Charge at {c_rate}C for {MAX_CHARGE_TIME_S} seconds"],
        "experiment_labels": [f"{c_rate}C"],
    }
    
    try:
        res = run_spmet(cell_design, config)
        if res[0]['success']:
            all_results[c_rate] = res[0]
            charge_time = (res[0]['time_s'][-1] - res[0]['time_s'][0]) / 60
            print(f"  {c_rate}C: {charge_time:.1f} min")
        else:
            print(f"  {c_rate}C: Failed - {res[0].get('error', 'Unknown')[:50]}")
    except Exception as e:
        print(f"  {c_rate}C: Error - {str(e)[:50]}")

In [ ]:
# Compare C-rates
if len(all_results) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(all_results)))
    
    for (c_rate, data), color in zip(all_results.items(), colors):
        time_min = (data['time_s'] - data['time_s'][0]) / 60
        
        # Voltage
        axes[0, 0].plot(time_min, data['voltage_V'], color=color, 
                        linewidth=2, label=f'{c_rate}C')
        
        # Current
        axes[0, 1].plot(time_min, -data['current_A'], color=color, 
                        linewidth=2, label=f'{c_rate}C')
        
        # Anode potential
        axes[1, 0].plot(time_min, data['anode_potential_V'], color=color, 
                        linewidth=2, label=f'{c_rate}C')
        
        # Temperature
        axes[1, 1].plot(time_min, data['temperature_K'] - 273.15, color=color, 
                        linewidth=2, label=f'{c_rate}C')
    
    axes[0, 0].set_xlabel('Time [min]')
    axes[0, 0].set_ylabel('Voltage [V]')
    axes[0, 0].set_title('Voltage vs Time')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].set_xlabel('Time [min]')
    axes[0, 1].set_ylabel('Charge Current [A]')
    axes[0, 1].set_title('Charge Current vs Time')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].set_xlabel('Time [min]')
    axes[1, 0].set_ylabel('Anode Potential [V]')
    axes[1, 0].set_title('Anode Potential vs Time')
    axes[1, 0].axhline(y=ANODE_POTENTIAL_THRESHOLD_V, color='r', linestyle='--', alpha=0.5)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].set_xlabel('Time [min]')
    axes[1, 1].set_ylabel('Temperature [C]')
    axes[1, 1].set_title('Temperature vs Time')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(f'Fast Charge Comparison: Multiple C-rates from {INITIAL_SOC*100:.0f}% SOC', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Export Charge Profile

In [ ]:
if r['success']:
    # Create DataFrame with charge profile
    charge_profile_df = pd.DataFrame({
        'time_s': time_s - time_s[0],
        'time_min': (time_s - time_s[0]) / 60,
        'voltage_V': voltage_V,
        'current_A': current_A,
        'charge_current_A': -current_A,
        'anode_potential_V': anode_potential_V,
        'temperature_C': temperature_K - 273.15,
    })

    # Display summary
    print("Charge Profile Summary:")
    print(charge_profile_df.describe())

    # Optionally save to CSV
    # charge_profile_df.to_csv('fast_charge_profile.csv', index=False)